# Communication Function Annotation Notebook (Gemma API, 5 Dimensions)

This notebook uses an API-style annotation loop. It annotates ads with Gemma only, saves checkpoints, keeps raw outputs and records parse/runtime failures without using fallback labels.

In [1]:
#%pip install -U pandas requests

### Imports 

In [2]:
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

### Data loading if you are running on Colab

### I am not using *colab* so mine are comment (please use command + / if you are using mac to uncomment them)

In [3]:
# #this try and except is only useful if you are running the code in both colab or locally, you do not have to comment it neighther way and please add the csv to drive if you use colab
# try:
#     from google.colab import drive  # type: ignore
#     drive.mount('/content/drive', force_remount=False)
# except Exception:
#     pass

# #this fun help you proceed further if you have colab or you run the code locally
# def find_project_root() -> Path:
#   #this checks if you are in colab
#     running_in_colab = 'google.colab' in sys.modules
#     candidates = []
#     if running_in_colab:
#         candidates.append(Path('/content/drive/MyDrive/Gliner-Work.Dauphine'))
#     candidates.append(Path('/Users/raresolteanu/Desktop/Gliner-Work.Dauphine'))

#     for candidate in candidates:
#         if (candidate / 'annotation_working_master_human_2100_seed.csv').exists():
#             return candidate
# #this checks if it is not in the colab it runs it locally
#     here = Path.cwd()
#     for candidate in [here, *here.parents]:
#         if (candidate / 'annotation_working_master_human_2100_seed.csv').exists():
#             return candidate
# #if none were found we set an error to rise
#     raise FileNotFoundError('Could not locate annotation_working_master_human_2100_seed.csv')


# PROJECT_ROOT = find_project_root() #call the fun
# DATASET_PATH = PROJECT_ROOT / 'annotation_working_master_human_2100_seed.csv' #the path to your CSV
# OUTPUT_DIR = PROJECT_ROOT / 'communication_function_outputs_5d'#output path
# OUTPUT_DIR.mkdir(parents=True, exist_ok=True) #do not crash if folder already exists

# #to see if everything work
# print('PROJECT_ROOT =', PROJECT_ROOT)
# print('DATASET_PATH exists =', DATASET_PATH.exists())
# print('OUTPUT_DIR =', OUTPUT_DIR)


### This is the data loading code if both the datasets and your notebook are in the same directory and if you are running it *locally*

In [4]:
PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / "annotation_working_master_human_2100_seed.csv"
OUTPUT_DIR = PROJECT_ROOT / "communication_function_outputs_gemma_api_5d"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATASET_PATH exists =", DATASET_PATH.exists())
print("OUTPUT_DIR =", OUTPUT_DIR)


PROJECT_ROOT = /Users/raresolteanu/Desktop/Gliner-Work.Dauphine
DATASET_PATH exists = True
OUTPUT_DIR = /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d


In [5]:
#the columns I want to use from the dataset to give as input to my annotater
TEXT_COLUMNS = [
    'Script',
    'Visuel',
    'Titre',
    'MotsClés',
    'Thème',
]

#the dimnsion that I want to use for annotation
DIMENSIONS = [
    'informativeness',
    'expressiveness',
    'phatic',
    'greenness',
    'creativeness',
]

#the scale
SCORE_MIN = 0.0
SCORE_MAX = 5.0
#this is in case of the dominant dimension if two have a score nearly equal the dominant dimension will be mixed
DOMINANT_TIE_TOLERANCE = 0.10

#change the modelm but make sure it is installed on ollama also by using the terminal(you can check with ollama list to see what you have install on it)

MODEL_NAME = "gemma3:1b" #this is the ligthest gemmav3 version


#the API for ollama
API_URL = 'http://localhost:11434/api/chat'

#This sets the maximum time, in seconds, that the notebook will wait for one API response from Ollama. 
#This needs to be incraesed if you are using more expensive models since more expensive models need more time and you will get to many time errors
REQUEST_TIMEOUT = 120

#this saves the progress every 10 rows
CHECKPOINT_EVERY = 10

#This is the threshold used to decide whether a row should be flagged as low-confidence.
CONFIDENCE_THRESHOLD = 0.0

#quick check of what we did above in this code box 
print('MODEL_NAME =', MODEL_NAME)
print('API_URL =', API_URL)
print('CHECKPOINT_EVERY =', CHECKPOINT_EVERY)

MODEL_NAME = gemma3:1b
API_URL = http://localhost:11434/api/chat
CHECKPOINT_EVERY = 10


This handles the missing values from NaN -> "", if not NaN it splits and strips it

In [6]:
def normalize_text(value):
    if pd.isna(value):
        return ''
    return ' '.join(str(value).split()).strip()

It trims long text before sending it to the model.

In [7]:
def shorten_text(text, max_chars=500):
    text = normalize_text(text)
    if len(text) <= max_chars: #if below 500 it returns it unchanged 
        return text
    return text[:max_chars].rstrip() + ' ...' #keeps only the charcters until the maxchar and put ... to see that the text was cut


This loads the dataset and keeps only what we want

In [8]:
def load_rows(limit=None, review_status='needs_label', row_ids=None):
    #this loads the data and make it numeric
    df = pd.read_csv(DATASET_PATH)
    df['row_id'] = pd.to_numeric(df['row_id'], errors='coerce').astype('Int64')
#keeps only rows where we have review status
    if review_status is not None and 'Review_Status' in df.columns:
        df = df[df['Review_Status'] == review_status].copy()
#this can be comment if you do not need to work with specific rows
    if row_ids is not None:
        wanted = {int(x) for x in row_ids}
        df = df[df['row_id'].isin(wanted)].copy()
#to set the limit
    if limit is not None:
        df = df.head(limit).copy()

    return df

This function checks whether the dataset contains all the columns you expect. This helps to make sure the models can see all the data I want above

In [9]:
def require_columns(df, columns):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(f'Missing columns in dataset: {missing}')


This function builds the final ad text that you send to the annotator. Plus it is using the shorten fun to reduce the input

In [10]:
def build_ad_text(row):
    parts = []
#this sets the length of the input
    for col, max_chars in [('Script', 280), ('Visuel', 180), ('Titre', 80), ('MotsClés', 80), ('Thème', 80)]:
#this gets the shorten value or gets '' if there is nothing
        value = shorten_text(row.get(col, ''), max_chars=max_chars)
        if value:
#this make the input dict
            parts.append(f'{col}: {value}')
#this joins the lines together
    return '\n'.join(parts)

In [11]:
preview = load_rows(limit=2, review_status='needs_label')
require_columns(preview, ['row_id'] + TEXT_COLUMNS)
preview[['row_id', 'Marque', 'Produit']].head()
preview = load_rows(limit=2, review_status='needs_label')
#check if we have all the columns
require_columns(preview, ['row_id'] + TEXT_COLUMNS)
#just data visualisation 
preview[['row_id', 'Produit','Script','Visuel','Titre','MotsClés','Thème']].head()


,row_id,Produit,Script,Visuel,Titre,MotsClés,Thème
2100,2100,NISSAN PRESTIGE,"Voix homme : "" NISSAN est fier de vous présent...","Sergio Aguero, nouvel ambassadeur de NISSAN, a...",SERGIO AGUERO,"LIGNE BLANCHE , PIEGE , PREVENTION , DEPASSEME...","AVENTURE, FANTASTIQUE, FOOTBALL, SECURITE, SUC..."
2101,2101,VW PRESTIGE,"Voix homme (1) : "" Il n'y a rien de tel que ri...",De la naissance de l'Univers à l'art contempor...,RIEN,"PUNK , RÉFRIGÉRATEUR , ADOLESCENT , TIMIDITÉ ,...","SEDUCTION, ESPACE, INNOVATION, LANGAGE, ANGLAI..."


how I want my output (JSON) to look like from the annotater

In [12]:
SCHEMA = {
    'informativeness': 'float 0.0 to 5.0',
    'expressiveness': 'float 0.0 to 5.0',
    'phatic': 'float 0.0 to 5.0',
    'greenness': 'float 0.0 to 5.0',
    'creativeness': 'float 0.0 to 5.0',
    'dominant_dimension': 'one of informativeness, expressiveness, phatic, greenness, creativeness, mixed',
    'dominant_dimension_score': 'float 0.0 to 5.0',
    'reason': 'short explanation under 12 words',
    'confidence': 'float 0.0 to 1.0'
}

The actual prompt

In [13]:
SYSTEM_PROMPT = f"""You are an expert annotator of automotive advertisements.
You score five communication dimensions.

Return only one valid JSON object.
Do not write markdown.
Do not use code fences.
Do not write any text before or after the JSON.

Schema:
{json.dumps(SCHEMA, ensure_ascii=False, indent=2)}

Rules:
- All five scores must be floats from 0.0 to 5.0
- dominant_dimension must be exactly one of: informativeness, expressiveness, phatic, greenness, creativeness, mixed
- dominant_dimension_score must equal the highest score
- confidence must be a float from 0.0 to 1.0
- reason must be short
"""

this function builds the user message sent to the model by placing the ad text inside a simple annotation prompt.

In [14]:
def build_user_prompt(row):
    return f"""Annotate this advertisement.

Ad:
{build_ad_text(row)}
"""

In [15]:
def extract_json(text):
    text = str(text).strip()
#this is looking for JSON object inside a fenced code block
    fenced = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', text, re.DOTALL)
    if fenced:
        try:
            return json.loads(fenced.group(1))
        except Exception:
            pass

    match = re.search(r'\{.*\}', text, re.DOTALL)


#this tries to parse that whole block as JSON.

    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            pass

    return None

In case the models gives different names for the columns 

In [16]:
def normalize_dimension_name(value):
    text = str(value or '').strip().lower()
    aliases = {
        'informative': 'informativeness',
        'referential': 'informativeness',
        'expressive': 'expressiveness',
        'emotive': 'expressiveness',
        'phatique': 'phatic',
        'green': 'greenness',
        'greeness': 'greenness',
        'ecology': 'greenness',
        'ecological': 'greenness',
        'creative': 'creativeness',
        'creativity': 'creativeness',
        'originality': 'creativeness',
    }
    text = aliases.get(text, text)
    return text if text in DIMENSIONS or text == 'mixed' else ''

This function makes sure a score is a valid number between your minimum and maximum.

In [17]:
def clamp_score(value):
    try:
        score = float(value) #this converts it in floot
    except Exception:
        score = SCORE_MIN #####should ask the supervisor if that is good or it should be change to NaN
#this forces the results between min and max
    return max(SCORE_MIN, min(SCORE_MAX, score)) 

This function decides which dimension is dominant.

In [18]:
def pick_dominant_dimension(scores):
  #this sorts it
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    top_label, top_score = ranked[0]

  #the rest of code checks the mixed cases
    same_top = [label for label, score in ranked if abs(score - top_score) <= DOMINANT_TIE_TOLERANCE]
    if len(same_top) > 1:
        return 'mixed', top_score
    return top_label, top_score

This function takes the parsed JSON from the model and cleans it.

In [19]:
def validate_payload(parsed):
    out = {}
#this loops through all dim and for each one gets the model output and it makes it between 0.0 and 5.0
    for dim in DIMENSIONS:
        out[dim] = clamp_score(parsed.get(dim))

#this picks a dominant dim
    inferred_dim, inferred_score = pick_dominant_dimension({dim: out[dim] for dim in DIMENSIONS})
#this take into account diff names 
    dominant_dimension = normalize_dimension_name(parsed.get('dominant_dimension')) or inferred_dim
#sets all scores between 0.0 and 5.0
    dominant_dimension_score = clamp_score(parsed.get('dominant_dimension_score'))
#checks whether the model’s dominant score matches the highest actual dimension score. 
    if abs(dominant_dimension_score - max(out[d] for d in DIMENSIONS)) > DOMINANT_TIE_TOLERANCE:
        dominant_dimension_score = inferred_score

#outputs
    out['dominant_dimension'] = dominant_dimension
    out['dominant_dimension_score'] = dominant_dimension_score
    out['reason'] = ' '.join(str(parsed.get('reason', '')).split()).strip()[:200]

#confidence this can be deleted or commented if you are not using confidence
    try:
        out['confidence'] = max(0.0, min(1.0, float(parsed.get('confidence', 0.0))))
    except Exception:
        out['confidence'] = 0.0

    return out

A function that annotates one dataset row using Gemma through the API.

In [20]:
def annotate_row_with_gemma(row):
#builds the chat messages sent to the model.The prompt
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_prompt(row)},
    ]

#Sends an HTTP POST request to the Ollama API endpoint.
    resp = requests.post(
        API_URL,
        json={
            'model': MODEL_NAME,
            'messages': messages,
            'stream': False, #asks for one full response, not streamed chunks
            'options': {'temperature': 0.0}, #deterministic output
        },
        timeout=REQUEST_TIMEOUT, #Sets how long Python waits before giving up on the request.
    )
    resp.raise_for_status()

#reads the API response as JSON and extracts the model’s generated text.
    generated = resp.json()['message']['content'] #this is the raw answer from Gemma
#this tries to extract a JSON object from the generated text.
    parsed = extract_json(generated)
    if parsed is None:
#for errors, failed rows are kept for inspection.
        return {
            'row_id': int(row['row_id']),
            'parse_ok': False,
            'error': 'Could not parse JSON',
            'raw_output': generated,
        }

#output
    validated = validate_payload(parsed)
    validated['row_id'] = int(row['row_id'])
    validated['model_name'] = MODEL_NAME
    validated['parse_ok'] = True
    validated['raw_output'] = generated
    validated['flagged'] = validated['confidence'] < CONFIDENCE_THRESHOLD
    return validated

### Quick sanity check

In [21]:
#loads one row from the dataset.
example = load_rows(limit=1, review_status='needs_label').iloc[0]
example_result = annotate_row_with_gemma(example)
#this keeps everything as dict besides the raw answer which is not useful for annotation
{k: v for k, v in example_result.items() if k != 'raw_output'}

{'informativeness': 4.5,
 'expressiveness': 4.0,
 'phatic': 4.0,
 'greenness': 3.0,
 'creativeness': 0.0,
 'dominant_dimension': 'expressiveness',
 'dominant_dimension_score': 4.5,
 'reason': 'The ad uses a compelling celebrity (Sergio Aguero) and a dynamic setting (football) to create a sense of excitement and adventure.',
 'confidence': 0.9,
 'row_id': 2100,
 'model_name': 'gemma3:1b',
 'parse_ok': True,
 'flagged': False}

The main pipeline function.

In [ ]:
def run_annotation_pipeline(limit=30, review_status='needs_label'):
#loads the rows you want to annotate.
    source_rows = load_rows(limit=limit, review_status=review_status).copy().reset_index(drop=True)
    total = len(source_rows)

#defines the output file paths.
    checkpoint_path = OUTPUT_DIR / 'gemma_api_checkpoint.csv'
    final_path = OUTPUT_DIR / 'gemma_api_annotations.csv'
    error_path = OUTPUT_DIR / 'gemma_api_errors.csv'

    results = []
    errors = []

#stores the start time of the whole run.
#this is used later to compute average time and ETA.
    run_start = time.time()

    for i, row in source_rows.iterrows():
        t0 = time.time() #stores the start time
#tries to annotate the row with Gemma.      
        try:
            result = annotate_row_with_gemma(row)
            if result.get('parse_ok'): #only if the row was parsed successfully:
                results.append(result)
            else: #if not add it to errors
                errors.append(result)
        except Exception as exc:
#adds a structured error record for that row.
            errors.append({
                'row_id': int(row['row_id']),
                'parse_ok': False,
                'error': str(exc),
                'raw_output': '',
            })
#computes how long this row took.
        elapsed = time.time() - t0
        done = i + 1 #counts how many rows have been processed so far.
        avg = (time.time() - run_start) / done #avg of them
        eta = avg * (total - done) / 60 if done < total else 0 #estimates the remaining time in minutes.
#prints live progress information, for example:    
        print(f'{done}/{total} {elapsed:.1f}s avg {avg:.1f}s ETA {eta:.1f} min')

#every CHECKPOINT_EVERY rows, save progress.
        if done % CHECKPOINT_EVERY == 0:
            pd.DataFrame(results).to_csv(checkpoint_path, index=False)
            pd.DataFrame(errors).to_csv(error_path, index=False)
            print(f'checkpoint saved at {done}/{total}')

#converts them into DF
    results_df = pd.DataFrame(results)
    errors_df = pd.DataFrame(errors)

    results_df.to_csv(final_path, index=False)
    errors_df.to_csv(error_path, index=False)

#prints the output file locations.
    print('wrote', final_path)
    print('wrote', error_path)
    return results_df, errors_df

#Then this line actually runs the pipeline:
results_df, errors_df = run_annotation_pipeline(limit=5, review_status='needs_label')
results_df.head()

#this line actually runs the pipeline
results_df, errors_df = run_annotation_pipeline(limit=5, review_status='needs_label')
results_df.head()

1/5 2.4s avg 2.4s ETA 0.2 min
2/5 2.2s avg 2.3s ETA 0.1 min
3/5 2.1s avg 2.2s ETA 0.1 min
4/5 2.1s avg 2.2s ETA 0.0 min
5/5 2.2s avg 2.2s ETA 0.0 min
wrote /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d/gemma_api_annotations.csv
wrote /Users/raresolteanu/Desktop/Gliner-Work.Dauphine/communication_function_outputs_gemma_api_5d/gemma_api_errors.csv


,informativeness,expressiveness,phatic,greenness,creativeness,dominant_dimension,dominant_dimension_score,reason,confidence,row_id,model_name,parse_ok,raw_output,flagged
0,4.5,4.0,4.0,3.0,0.0,expressiveness,4.5,The ad uses a compelling celebrity (Sergio Agu...,0.9,2100,gemma3:1b,True,"```json\n{\n ""informativeness"": 4.5,\n ""expr...",False
1,4.5,4.0,3.0,3.0,0.0,creativeness,4.5,The ad uses evocative language and imagery to ...,0.9,2101,gemma3:1b,True,"```json\n{\n ""informativeness"": 4.5,\n ""expr...",False
2,5.0,4.0,3.0,2.0,0.0,informativeness,5.0,The ad highlights the product's usefulness and...,0.9,2102,gemma3:1b,True,"```json\n{\n ""informativeness"": 5.0,\n ""expr...",False
3,5.0,4.0,4.0,3.0,0.0,expressiveness,5.0,The ad uses evocative language and imagery to ...,0.9,2103,gemma3:1b,True,"```json\n{\n ""informativeness"": 5.0,\n ""expr...",False
4,4.5,4.0,4.0,3.0,0.0,expressiveness,4.5,The ad uses engaging language and a relatable ...,0.9,2104,gemma3:1b,True,"```json\n{\n ""informativeness"": 4.5,\n ""expr...",False
